In [1]:
!pip -q install -U transformers
import os, re, math, random, numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print(dev)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "aubmindlab/aragpt2-base"   # Arabic GPT-2. Alternative: "gpt2" (English; Arabic output will be poor)

cuda


In [2]:
CSV = "/kaggle/input/datasets/laghrissimehdi/dataset-education/dataset_education.csv"
df = pd.read_csv(CSV)
clean = lambda t: re.sub(r"\s+", " ", str(t)).strip()

train_txt = [clean(t) for t in df[(df.label == 2) & (df.split == "train")]["Text"]]
val_txt   = [clean(t) for t in df[(df.label == 2) & (df.split == "val")]["Text"]]
print(len(train_txt), "train texts,", len(val_txt), "validation texts")

# the customized dataset, saved as a text file (one text per line)
with open("education_corpus.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(train_txt))
print(sum(len(t.split()) for t in train_txt), "words in the training corpus")

63 train texts, 13 validation texts
29761 words in the training corpus


In [3]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None: tok.pad_token = tok.eos_token
BLOCK = 256

def make_blocks(texts):
    ids = []
    for t in texts: ids += tok.encode(t) + [tok.eos_token_id]
    n = len(ids) // BLOCK
    return torch.tensor(ids[:n * BLOCK]).view(n, BLOCK)

train_blocks, val_blocks = make_blocks(train_txt), make_blocks(val_txt)
print("train blocks:", tuple(train_blocks.shape), "| val blocks:", tuple(val_blocks.shape))

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

train blocks: (165, 256) | val blocks: (36, 256)


In [4]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(dev)

@torch.no_grad()
def perplexity(blocks, bs=8):
    model.eval(); tot = n = 0
    for i in range(0, len(blocks), bs):
        x = blocks[i:i+bs].to(dev)
        tot += model(x, labels=x).loss.item() * x.size(0); n += x.size(0)
    return math.exp(tot / n)

def generate(prompt, n=1, max_new=120):
    model.eval()
    enc = tok(prompt, return_tensors="pt").to(dev)
    out = model.generate(**enc, do_sample=True, top_k=50, top_p=0.95, temperature=0.9,
                         repetition_penalty=1.2, no_repeat_ngram_size=3,
                         max_new_tokens=max_new, num_return_sequences=n,
                         pad_token_id=tok.pad_token_id)
    return [tok.decode(o, skip_special_tokens=True) for o in out]

PROMPTS = [
    "تعاني المنظومة التعليمية في المغرب من",
    "أعلنت وزارة التربية الوطنية عن",
    "يعتبر الهدر المدرسي من أبرز",
    "يواجه طلبة الجامعات المغربية",
]
ppl_before = perplexity(val_blocks); print("perplexity BEFORE fine-tuning:", round(ppl_before, 1))
gen_before = {p: generate(p)[0] for p in PROMPTS}
for p, g in gen_before.items(): print("\n[BEFORE]", g)

model.safetensors: reconstructing file:   0%|          |  0.00B /  553MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: aubmindlab/aragpt2-base
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


perplexity BEFORE fine-tuning: 134.4

[BEFORE] تعاني المنظومة التعليمية في المغرب من صعوبات كبيرة ، منها العجز عن مواكبة التحولات التي تشهدها هذه المرحلة ، وغياب روح المبادرة لدى المدرسين والطلبة ، وقلة الاهتمام بالمجال التربوي على غرار ما هو عليه الأمر بالنسبة للتعليم الخاص والتعليم الخصوصي . كما أن عدم وجود مؤسسات تربوية قوية وفاعلة كفيلة بجعل التلميذات يتلقين جميع العلوم والمعارف والمهارات اللازمة ، مما يحول دون تحقيق مستوى أفضل من التمدرس إلى جانب انخفاض مستوى التنمية البشرية بصفة عامة .انطلقت دورة هذا العام 2015 بعد غياب عشر سنوات . وقد تمكنت من تجاوز عدد الأساتذة المسجلين الذين استفادوا من دورات سابقة خلال السنوات السبع الأخيرة من بلوغ 100 أستاذ جديد في تخصصات عديدة ومتنوعة ، مع التركيز أيضا على تدريس

[BEFORE] أعلنت وزارة التربية الوطنية عن فتح باب التسجيل القبلي للطلبة الحاصلين على الاجازة المهنية أو ما يعادلها من داخل المملكة المغربية والذين يتابعون دراستهم بالقسم الداخلي في التعليم العالي والبحث العلمي .وطالبت الوزارة جميع الطلبة المسجلين في المدارس العليا للتعليم ( الإجازة ا

In [5]:
EPOCHS, BS, LR = 4, 4, 5e-5
dl = DataLoader(train_blocks, batch_size=BS, shuffle=True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total = len(dl) * EPOCHS
sch = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
use_amp = dev.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

history = []
for ep in range(EPOCHS):
    model.train(); run = 0
    for x in dl:
        x = x.to(dev)
        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
            loss = model(x, labels=x).loss
        opt.zero_grad()
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sch.step()
        run += loss.item()
    tr_loss, vp = run / len(dl), perplexity(val_blocks)
    history.append(dict(epoch=ep + 1, train_loss=tr_loss, val_ppl=vp))
    print(f"epoch {ep+1}/{EPOCHS}  train loss {tr_loss:.3f}  val perplexity {vp:.1f}")

/tmp/ipykernel_50/1155669348.py:19: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sch.step()


epoch 1/4  train loss 7.196  val perplexity 92.7
epoch 2/4  train loss 5.535  val perplexity 77.5
epoch 3/4  train loss 5.076  val perplexity 71.1
epoch 4/4  train loss 4.862  val perplexity 69.8


In [6]:
ppl_after = perplexity(val_blocks)
print(f"\nPerplexity on validation texts: {ppl_before:.1f} -> {ppl_after:.1f}")

rows = []
for p in PROMPTS:
    g = generate(p, max_new=150)[0]
    rows.append(dict(prompt=p, before=gen_before[p], after=g))
    print("\n[PROMPT]", p, "\n[AFTER] ", g)

pd.DataFrame(rows).to_csv("generations.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(history).to_csv("gpt2_training_history.csv", index=False)
model.save_pretrained("gpt2_education"); tok.save_pretrained("gpt2_education")


Perplexity on validation texts: 134.4 -> 69.8

[PROMPT] تعاني المنظومة التعليمية في المغرب من 
[AFTER]  تعاني المنظومة التعليمية في المغرب من ضعف التحصيل الدراسي، وصعوبة التعلم؛ وانعدام وسائل الاتصال والتواصل ، وارتفاع نسب الهدر المدرسي. كما يشكل التلاميذ عبئا اجتماعيا ثقيلا على أداء الواجبات المدرسية، وعدم فهم طبيعة العمل داخل الفصول الدراسية إلى جانب قلة النوم”.

[PROMPT] أعلنت وزارة التربية الوطنية عن 
[AFTER]  أعلنت وزارة التربية الوطنية عن فتح مباراة توظيف أساتذة مساعدين في مناصب الدرجة الثالثة. ودعا المكتب الوطني للتعليم الأولي أن يكون موعد الدخول إلى أجل اجتياز هذه المباراة طبقا لمقتضيات النظام الأساسي للوظيفة العمومية، داعيا إلى اعتماد الجدول المحدد لضمان تكافؤ الفرص لاجتياز الامتحانات بالنسبة للأساتذة المرشحين الحاصلين على شهادة التأهيل العالي؛ وكذا ضمان مواصلة انخراطهم من طرف الأساتذة الذين لم يلتحقوا بعد برسم الموسم الدراسي المقبل-ولفت البلاغ ذاته بأن “تخضع بعض المترشحين المقبولين خلال العطلة الجامعية الحالية في الآجال المحددة سلفا”. وأشار المصدر نفسه إلى أنه يتعين تعبئة ال

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('gpt2_education/tokenizer_config.json', 'gpt2_education/tokenizer.json')